###Day 17 — Objective

To train and evaluate a comparable object detection model on enhanced underwater images using White Balance and CLAHE,and compare its detection performance with the original-image baseline across all three supplied datasets.

In [ ]:
from google.colab import drive
import torch
drive.mount("/content/drive")
print("Google Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [ ]:
import os
import shutil
SOURCE="/content/drive/MyDrive/Dataset_V1_Enhanced"
DEST="/content/Dataset_V1_Enhanced"
if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(SOURCE,DEST)
print("Enhanced dataset copied successfully!")
print("Source:",SOURCE)
print("Local path:",DEST)

Enhanced dataset copied successfully!
Source: /content/drive/MyDrive/Dataset_V1_Enhanced
Local path: /content/Dataset_V1_Enhanced


In [ ]:
import os
base="/content/Dataset_V1_Enhanced"
datasets=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in datasets:
    print("\n",dataset)
    path=os.path.join(base,dataset)
    for split in ["train","valid","test"]:
        images=os.path.join(path,split,"images")
        labels=os.path.join(path,split,"labels")
        print(split,
              "images:",os.path.exists(images),
              "labels:",os.path.exists(labels))


 DIATAquarium.v4i.yolov8
train images: True labels: True
valid images: True labels: True
test images: False labels: False

 Aquatic Plant.v2i.yolov8
train images: True labels: True
valid images: False labels: False
test images: False labels: False

 well.v8i.yolov8
train images: True labels: True
valid images: False labels: False
test images: False labels: False


In [ ]:
import os
for dataset in [
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]:
    print("\n"+dataset)
    path=f"/content/Dataset_V1/{dataset}"
    for split in ["train","valid","test"]:
        split_path=os.path.join(path,split)
        print(split,os.path.exists(split_path))


DIATAquarium.v4i.yolov8
train False
valid False
test False

Aquatic Plant.v2i.yolov8
train False
valid False
test False

well.v8i.yolov8
train False
valid False
test False


In [ ]:
import os
print("Folders inside /content:")
for item in os.listdir("/content"):
    if os.path.isdir(os.path.join("/content",item)):
        print(item)

Folders inside /content:
.config
drive
Dataset_V1_Enhanced
sample_data


In [ ]:
import os
original="/content/drive/MyDrive/Dataset_V1"
print("Dataset_V1 exists:",os.path.exists(original))
if os.path.exists(original):
    print("\nDatasets found:")
    for item in os.listdir(original):
        if os.path.isdir(os.path.join(original,item)):
            print(item)

Dataset_V1 exists: True

Datasets found:
Aquatic Plant.v2i.yolov8
well.v8i.yolov8
split_plan
DIATAquarium.v4i.yolov8


In [ ]:
import os
base="/content/drive/MyDrive/Dataset_V1"
datasets=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in datasets:
    print("\n"+dataset)
    path=os.path.join(base,dataset)
    for split in ["train","valid","test"]:
        images=os.path.join(path,split,"images")
        labels=os.path.join(path,split,"labels")
        print(
            split,
            "images:",os.path.exists(images),
            "labels:",os.path.exists(labels)
        )


DIATAquarium.v4i.yolov8
train images: True labels: True
valid images: True labels: True
test images: False labels: False

Aquatic Plant.v2i.yolov8
train images: True labels: True
valid images: True labels: True
test images: True labels: True

well.v8i.yolov8
train images: True labels: True
valid images: True labels: True
test images: True labels: True


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
ORIGINAL="/content/drive/MyDrive/Dataset_V1"
ENHANCED="/content/Dataset_V1_Enhanced"
def white_balance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    avg_a=np.mean(a)
    avg_b=np.mean(b)
    a=np.clip(a-(avg_a-128)*l/255,0,255).astype(np.uint8)
    b=np.clip(b-(avg_b-128)*l/255,0,255).astype(np.uint8)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)
def apply_clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=clahe.apply(l)
    enhanced=cv2.merge((l,a,b))
    return cv2.cvtColor(enhanced,cv2.COLOR_LAB2BGR)
def enhance_image(img):
    img=white_balance(img)
    img=apply_clahe(img)
    return img
datasets=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in datasets:
    print("\nProcessing:",dataset)
    for split in ["valid","test"]:
        src_images=os.path.join(ORIGINAL,dataset,split,"images")
        src_labels=os.path.join(ORIGINAL,dataset,split,"labels")
        if not os.path.exists(src_images):
            print(split,": not available")
            continue
        dst_images=os.path.join(ENHANCED,dataset,split,"images")
        dst_labels=os.path.join(ENHANCED,dataset,split,"labels")
        os.makedirs(dst_images,exist_ok=True)
        os.makedirs(dst_labels,exist_ok=True)
        image_files=[
            f for f in os.listdir(src_images)
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
        ]

        for filename in tqdm(image_files,desc=split):
            src=os.path.join(src_images,filename)
            dst=os.path.join(dst_images,filename)
            img=cv2.imread(src)
            if img is None:
                print("Could not read:",filename)
                continue
            enhanced=enhance_image(img)
            cv2.imwrite(dst,enhanced)
        label_files=[
            f for f in os.listdir(src_labels)
            if f.lower().endswith(".txt")
        ]
        for filename in label_files:
            src=os.path.join(src_labels,filename)
            dst=os.path.join(dst_labels,filename)
            if not os.path.exists(dst):
                import shutil
                shutil.copy2(src,dst)
        print(split,"completed:",len(image_files),"images")


Processing: DIATAquarium.v4i.yolov8


valid: 100%|██████████| 773/773 [02:40<00:00,  4.82it/s]


valid completed: 773 images
test : not available

Processing: Aquatic Plant.v2i.yolov8


valid: 100%|██████████| 179/179 [00:08<00:00, 22.26it/s]


valid completed: 179 images


...

test completed: 89 images

Processing: well.v8i.yolov8


valid: 100%|██████████| 191/191 [00:09<00:00, 20.99it/s]


valid completed: 191 images


test: 100%|██████████| 184/184 [00:10<00:00, 17.76it/s]


test completed: 184 images


In [ ]:
import os
base="/content/Dataset_V1_Enhanced"
datasets=[
    "DIATAquarium.v4i.yolov8",
    "Aquatic Plant.v2i.yolov8",
    "well.v8i.yolov8"
]
for dataset in datasets:
    print("\n"+dataset)
    dataset_path=os.path.join(base,dataset)
    for split in ["train","valid","test"]:
        images_path=os.path.join(dataset_path,split,"images")
        labels_path=os.path.join(dataset_path,split,"labels")
        if not os.path.exists(images_path):
            print(split,": not available")
            continue
        images=[
            f for f in os.listdir(images_path)
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
        ]
        labels=[
            f for f in os.listdir(labels_path)
            if f.lower().endswith(".txt")
        ]
        image_names={os.path.splitext(f)[0] for f in images}
        label_names={os.path.splitext(f)[0] for f in labels}
        missing_labels=image_names-label_names
        extra_labels=label_names-image_names
        print(
            split,
            "| images:",len(images),
            "| labels:",len(labels),
            "| missing labels:",len(missing_labels),
            "| extra labels:",len(extra_labels)
        )


DIATAquarium.v4i.yolov8
train | images: 1410 | labels: 1462 | missing labels: 1156 | extra labels: 1208
valid | images: 773 | labels: 773 | missing labels: 0 | extra labels: 0
test : not available

Aquatic Plant.v2i.yolov8
train | images: 1867 | labels: 1876 | missing labels: 16 | extra labels: 25
valid | images: 179 | labels: 179 | missing labels: 0 | extra labels: 0
test | images: 89 | labels: 89 | missing labels: 0 | extra labels: 0

well.v8i.yolov8
train | images: 3782 | labels: 3768 | missing labels: 27 | extra labels: 13
valid | images: 191 | labels: 191 | missing labels: 0 | extra labels: 0
test | images: 184 | labels: 184 | missing labels: 0 | extra labels: 0


In [ ]:
import os
base="/content/Dataset_V1_Enhanced"
configs={
    "DIATAquarium.v4i.yolov8":{
        "nc":11,
        "names":[
            "Aquatic Plant",
            "Camera",
            "Conch-Shell",
            "Fish",
            "Fishes",
            "Pluco",
            "Shark",
            "Temperature Sensor",
            "Water-Pump",
            "Water-filter",
            "betta"
        ]
    },
    "Aquatic Plant.v2i.yolov8":{
        "nc":1,
        "names":["Aquatic_plant"]
    },
    "well.v8i.yolov8":{
        "nc":4,
        "names":[
            "Inlet-pipe",
            "fishes",
            "school-of-fish",
            "stone"
        ]
    }
}
for dataset,config in configs.items():
    yaml_path=os.path.join(base,dataset,"enhanced.yaml")
    lines=[
        f"path: {os.path.join(base,dataset)}",
        "train: train/images",
        "val: valid/images"
    ]
    if os.path.exists(os.path.join(base,dataset,"test","images")):
        lines.append("test: test/images")
    lines.append(f"nc: {config['nc']}")
    lines.append("names:")
    for i,name in enumerate(config["names"]):
        lines.append(f"  {i}: {name}")
    with open(yaml_path,"w") as f:
        f.write("\n".join(lines)+"\n")
    print("Created:",yaml_path)

Created: /content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml
Created: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/enhanced.yaml
Created: /content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml


In [ ]:
import os
base="/content/Dataset_V1_Enhanced"
datasets=[
"DIATAquarium.v4i.yolov8",
"Aquatic Plant.v2i.yolov8",
"well.v8i.yolov8"
]

for dataset in datasets:
    yaml_path=os.path.join(base,dataset,"enhanced.yaml")
    print("\n"+dataset)
    print("-"*40)
    with open(yaml_path,"r") as f:
        print(f.read())


DIATAquarium.v4i.yolov8
----------------------------------------
path: /content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8
train: train/images
val: valid/images
nc: 11
names:
  0: Aquatic Plant
  1: Camera
  2: Conch-Shell
  3: Fish
  4: Fishes
  5: Pluco
  6: Shark
  7: Temperature Sensor
  8: Water-Pump
  9: Water-filter
  10: betta


Aquatic Plant.v2i.yolov8
----------------------------------------
path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant


well.v8i.yolov8
----------------------------------------
path: /content/Dataset_V1_Enhanced/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone



In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
DIAT_YAML="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
data=DIAT_YAML,
epochs=30,
imgsz=640,
batch=16,
device=0,
project="/content/Day17_YOLO_Enhanced",
name="DIATAquarium"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'KDP-SCS645237-C.NITISH KUMAR-TEL.pdf', 'Classroom', 'Letter.gdoc', 'Dataset_V1_Enhanced (1)', 'Dataset_V1_Enhanced', 'Dataset_V1']


In [ ]:
import os
base="/content/drive/MyDrive/Dataset_V1/DIATAquarium.v4i.yolov8/train"
images=len([f for f in os.listdir(os.path.join(base,"images")) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
labels=len([f for f in os.listdir(os.path.join(base,"labels")) if f.lower().endswith(".txt")])
print("Original DIAT train images:",images)
print("Original DIAT train labels:",labels)

Original DIAT train images: 8121
Original DIAT train labels: 8121


In [ ]:
import os

for folder in ["Dataset_V1_Enhanced","Dataset_V1_Enhanced (1)"]:
    base=f"/content/drive/MyDrive/{folder}/DIATAquarium.v4i.yolov8/train/images"
    if os.path.exists(base):
        count=len([f for f in os.listdir(base) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
        print(folder,":",count,"images")
    else:
        print(folder,": DIAT folder not found")

Dataset_V1_Enhanced : 8121 images
Dataset_V1_Enhanced (1) : 8121 images


In [ ]:
import os

base="/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"

print(os.listdir(base))

['train', 'data.yaml']


In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm
import numpy as np

ORIGINAL="/content/drive/MyDrive/Dataset_V1/DIATAquarium.v4i.yolov8"
ENHANCED="/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"

src_images=os.path.join(ORIGINAL,"valid","images")
src_labels=os.path.join(ORIGINAL,"valid","labels")

dst_images=os.path.join(ENHANCED,"valid","images")
dst_labels=os.path.join(ENHANCED,"valid","labels")

os.makedirs(dst_images,exist_ok=True)
os.makedirs(dst_labels,exist_ok=True)

def white_balance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    avg_a=np.mean(a)
    avg_b=np.mean(b)
    a=np.clip(a-(avg_a-128)*l/255,0,255).astype(np.uint8)
    b=np.clip(b-(avg_b-128)*l/255,0,255).astype(np.uint8)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)

def apply_clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=clahe.apply(l)
    enhanced=cv2.merge((l,a,b))
    return cv2.cvtColor(enhanced,cv2.COLOR_LAB2BGR)

def enhance_image(img):
    img=white_balance(img)
    img=apply_clahe(img)
    return img

image_files=[f for f in os.listdir(src_images) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]

print("Validation images:",len(image_files))

for filename in tqdm(image_files,desc="Enhancing DIAT validation"):
    src=os.path.join(src_images,filename)
    dst=os.path.join(dst_images,filename)

    img=cv2.imread(src)

    if img is None:
        print("Could not read:",filename)
        continue

    enhanced=enhance_image(img)
    cv2.imwrite(dst,enhanced)

label_files=[f for f in os.listdir(src_labels) if f.lower().endswith(".txt")]

for filename in tqdm(label_files,desc="Copying validation labels"):
    shutil.copy2(
        os.path.join(src_labels,filename),
        os.path.join(dst_labels,filename)
    )

print("DIAT enhanced validation set completed.")
print("Images:",len(os.listdir(dst_images)))
print("Labels:",len(os.listdir(dst_labels)))

Validation images: 773


Copying validation labels: 100%|██████████| 773/773 [00:14<00:00, 53.15it/s]

DIAT enhanced validation set completed.
Images: 773
Labels: 773


In [ ]:
yaml_path="/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml"
yaml_content="""path: /content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8
train: train/images
val: valid/images
nc: 11
names:
  0: Aquatic Plant
  1: Camera
  2: Conch-Shell
  3: Fish
  4: Fishes
  5: Pluco
  6: Shark
  7: Temperature Sensor
  8: Water-Pump
  9: Water-filter
  10: betta
"""

with open(yaml_path,"w") as f:
    f.write(yaml_content)

print("Enhanced YAML created:")
print(yaml_path)

Enhanced YAML created:
/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml


In [ ]:
import os

base="/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"

for split in ["train","valid"]:
    image_dir=os.path.join(base,split,"images")
    label_dir=os.path.join(base,split,"labels")

    images=len([f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
    labels=len([f for f in os.listdir(label_dir) if f.lower().endswith(".txt")])

    print(split,"images:",images,"labels:",labels)

print("\nYAML:")
with open(os.path.join(base,"enhanced.yaml")) as f:
    print(f.read())

train images: 8121 labels: 8121
valid images: 773 labels: 773

YAML:
path: /content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8
train: train/images
val: valid/images
nc: 11
names:
  0: Aquatic Plant
  1: Camera
  2: Conch-Shell
  3: Fish
  4: Fishes
  5: Pluco
  6: Shark
  7: Temperature Sensor
  8: Water-Pump
  9: Water-filter
  10: betta



In [ ]:
import shutil
import os

local="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"

if os.path.exists(local):
    shutil.rmtree(local)

print("Incomplete local DIAT enhanced folder removed.")

Incomplete local DIAT enhanced folder removed.


In [ ]:
import os
import shutil

source="/content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"
destination="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"

print("Copying complete DIAT enhanced dataset...")
shutil.copytree(source,destination)
print("Copy completed.")

Copying complete DIAT enhanced dataset...
Copy completed.


In [ ]:
import os
base="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8"
for split in ["train","valid"]:
    image_dir=os.path.join(base,split,"images")
    label_dir=os.path.join(base,split,"labels")

    images=len([f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
    labels=len([f for f in os.listdir(label_dir) if f.lower().endswith(".txt")])

    print(split,"images:",images,"labels:",labels)

train images: 8121 labels: 8121
valid images: 773 labels: 773


In [ ]:
from ultralytics import YOLO
DIAT_ENHANCED_YAML="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
    data=DIAT_ENHANCED_YAML,
    epochs=20,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/Day17_YOLO_Enhanced",
    name="DIATAquarium",
    exist_ok=True
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=DIATAquari

In [ ]:
from ultralytics import YOLO
BEST_MODEL="/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/weights/best.pt"
DIAT_YAML="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml"
model=YOLO(BEST_MODEL)
results=model.val(
    data=DIAT_YAML,
    split="val",
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.1 ms, read: 200.6±34.2 MB/s, size: 466.1 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/12c80KPG7UpGhLH4F8MCROtxOtRrpjxRh/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/valid/labels.cache... 773 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 773/773 52.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 2.7it/s 18.5s
                   all        773       1504      0.876      0.901      0.911      0.639
         Aquatic Plant        358        409      0.973       0.99      0.985       0.79
                Camera         15         15      0.966      0.867      0.952      0.558
           Conch-Shell        143        159      0.916      0.956      0.945        0.7
                  Fish         70   

In [ ]:
names=model.names
print("DIAT Enhanced — Class-wise Results")
print("-"*75)
for i,name in names.items():
    print(
        f"{name}: "
        f"Precision={results.box.p[i]:.4f}, "
        f"Recall={results.box.r[i]:.4f}, "
        f"mAP50={results.box.ap50[i]:.4f}, "
        f"mAP50-95={results.box.ap[i].mean():.4f}"
    )

DIAT Enhanced — Class-wise Results
---------------------------------------------------------------------------
Aquatic Plant: Precision=0.9732, Recall=0.9902, mAP50=0.9847, mAP50-95=0.7896
Camera: Precision=0.9664, Recall=0.8667, mAP50=0.9523, mAP50-95=0.5577
Conch-Shell: Precision=0.9156, Recall=0.9557, mAP50=0.9453, mAP50-95=0.7005
Fish: Precision=0.8306, Recall=0.8194, mAP50=0.8677, mAP50-95=0.4677
Fishes: Precision=0.5445, Recall=0.7692, mAP50=0.6547, mAP50-95=0.3827
Pluco: Precision=0.8899, Recall=0.8978, mAP50=0.9190, mAP50-95=0.6827
Shark: Precision=0.9245, Recall=0.9582, mAP50=0.9711, mAP50-95=0.6514
Temperature Sensor: Precision=0.8129, Recall=0.9375, mAP50=0.9399, mAP50-95=0.6608
Water-Pump: Precision=0.9073, Recall=0.8808, mAP50=0.9400, mAP50-95=0.7293
Water-filter: Precision=0.9258, Recall=0.8571, mAP50=0.8889, mAP50-95=0.6762
betta: Precision=0.9435, Recall=0.9795, mAP50=0.9623, mAP50-95=0.7338


In [ ]:
import os
import time
VAL_DIR="/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/valid/images"
images=[
    os.path.join(VAL_DIR,f)
    for f in os.listdir(VAL_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
][:100]
start=time.time()
for image in images:
    model.predict(
        source=image,
        imgsz=640,
        device=0,
        verbose=False
    )
total_time=time.time()-start
avg_time=total_time/len(images)
fps=1/avg_time
print("Images measured:",len(images))
print("Total prediction time:",round(total_time,2),"seconds")
print("Average prediction time:",round(avg_time*1000,2),"ms/image")
print("Approximate FPS:",round(fps,2))

Images measured: 100
Total prediction time: 3.0 seconds
Average prediction time: 29.96 ms/image
Approximate FPS: 33.38


In [ ]:
PRED_DIR="/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/predictions"

model.predict(
    source=images[:10],
    imgsz=640,
    device=0,
    save=True,
    project=PRED_DIR,
    name="validation_predictions",
    exist_ok=True
)

print("Prediction images saved to:")
print(PRED_DIR)


0: 640x640 1 betta, 6.5ms
1: 640x640 1 Shark, 1 betta, 6.5ms
2: 640x640 1 Aquatic Plant, 1 Shark, 1 Water-Pump, 6.5ms
3: 640x640 1 Shark, 6.5ms
4: 640x640 1 Aquatic Plant, 1 Conch-Shell, 1 Water-Pump, 6.5ms
5: 640x640 1 Conch-Shell, 2 Sharks, 6.5ms
6: 640x640 2 Aquatic Plants, 1 Water-Pump, 6.5ms
7: 640x640 1 Aquatic Plant, 1 Water-Pump, 6.5ms
8: 640x640 1 Water-filter, 1 betta, 6.5ms
9: 640x640 1 Aquatic Plant, 1 Water-Pump, 6.5ms
Speed: 2.8ms preprocess, 6.5ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/predictions/validation_predictions
Prediction images saved to:
/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/predictions


In [ ]:
import os
model_path="/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/weights/best.pt"
size_mb=os.path.getsize(model_path)/(1024*1024)
print("Model size:",round(size_mb,2),"MB")
print("Model path:",model_path)

Model size: 5.95 MB
Model path: /content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/weights/best.pt


In [ ]:
summary="""# DIATAquarium — YOLOv8n Enhanced

## Dataset
- Training images: 8,121
- Validation images: 773
- Test set: Not available in the supplied local dataset
- Number of classes: 11

## Model Configuration
- Model: YOLOv8n
- Epochs: 20
- Image Size: 640 × 640
- Batch Size: 16
- Enhancement: White Balance + CLAHE

## Validation Performance

| Metric | Result |
|---|---:|
| Precision | 87.60% |
| Recall | 90.10% |
| mAP@0.5 | 91.10% |
| mAP@0.5:0.95 | 63.90% |

## Inference Performance
- Images measured: 100
- Total prediction time: 3.00 seconds
- Average prediction time: 29.96 ms/image
- Approximate FPS: 33.38
- Ultralytics inference component: 6.5 ms/image

## Model Information
- Model size: 5.95 MB
- Parameters: 3,007,793
- GFLOPs: 8.1

## Saved Outputs
- Best model: /content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/weights/best.pt
- Prediction outputs: /content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/predictions/validation_predictions

## Evaluation Note
The reported Precision, Recall and mAP values were obtained using the 773-image validation set. A separate local test set was not available for DIATAquarium. The enhanced experiment used 20 training epochs, whereas the original Day 16 baseline used 30 epochs. Therefore, the comparison should acknowledge the different training duration.

## Tracking
No video or sequential data was available. Therefore, tracking was not implemented.
"""

path="/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/DIATAquarium_Day17_Summary.md"

with open(path,"w") as f:
    f.write(summary)

print("Summary saved successfully!")
print(path)

Summary saved successfully!
/content/drive/MyDrive/Day17_YOLO_Enhanced/DIATAquarium/DIATAquarium_Day17_Summary.md


In [ ]:
import os

BASE="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8"

for split in ["train","valid","test"]:
    image_dir=os.path.join(BASE,split,"images")
    label_dir=os.path.join(BASE,split,"labels")

    if not os.path.exists(image_dir):
        print(split,": images folder NOT FOUND")
        continue

    images=[f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
    labels=[f for f in os.listdir(label_dir) if f.lower().endswith(".txt")] if os.path.exists(label_dir) else []

    print(split)
    print("Images:",len(images))
    print("Labels:",len(labels))
    print("Missing labels:",len(set(os.path.splitext(f)[0] for f in images)-set(os.path.splitext(f)[0] for f in labels)))
    print()

train
Images: 1892
Labels: 1879
Missing labels: 13

valid : images folder NOT FOUND
test : images folder NOT FOUND


In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
ORIGINAL="/content/drive/MyDrive/Dataset_V1"
ENHANCED="/content/Dataset_V1_Enhanced"
def white_balance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    avg_a=np.mean(a)
    avg_b=np.mean(b)
    a=np.clip(a-(avg_a-128)*l/255,0,255).astype(np.uint8)
    b=np.clip(b-(avg_b-128)*l/255,0,255).astype(np.uint8)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)
def apply_clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=clahe.apply(l)
    enhanced=cv2.merge((l,a,b))
    return cv2.cvtColor(enhanced,cv2.COLOR_LAB2BGR)
def enhance_image(img):
    return apply_clahe(white_balance(img))
dataset="Aquatic Plant.v2i.yolov8"
for split in ["valid","test"]:
    src_images=os.path.join(ORIGINAL,dataset,split,"images")
    src_labels=os.path.join(ORIGINAL,dataset,split,"labels")
    dst_images=os.path.join(ENHANCED,dataset,split,"images")
    dst_labels=os.path.join(ENHANCED,dataset,split,"labels")
    if not os.path.exists(src_images):
        print(split,"not available in original dataset")
        continue
    os.makedirs(dst_images,exist_ok=True)
    os.makedirs(dst_labels,exist_ok=True)
    image_files=[f for f in os.listdir(src_images) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
    for filename in tqdm(image_files,desc="Enhancing "+split):
        src=os.path.join(src_images,filename)
        dst=os.path.join(dst_images,filename)
        img=cv2.imread(src)
        if img is None:
            continue
        enhanced=enhance_image(img)
        cv2.imwrite(dst,enhanced)
    label_files=[f for f in os.listdir(src_labels) if f.lower().endswith(".txt")]
    for filename in label_files:
        src=os.path.join(src_labels,filename)
        dst=os.path.join(dst_labels,filename)
        shutil.copy2(src,dst)
    print(split,"completed:",len(image_files),"images")

Enhancing valid: 100%|██████████| 179/179 [00:08<00:00, 19.92it/s]


valid completed: 179 images


Enhancing test: 100%|██████████| 89/89 [00:41<00:00,  2.14it/s]


test completed: 89 images


In [ ]:
import os
BASE="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8"
for split in ["train","valid","test"]:
    image_dir=os.path.join(BASE,split,"images")
    label_dir=os.path.join(BASE,split,"labels")
    images=[f for f in os.listdir(image_dir) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
    labels=[f for f in os.listdir(label_dir) if f.lower().endswith(".txt")]
    image_names={os.path.splitext(f)[0] for f in images}
    label_names={os.path.splitext(f)[0] for f in labels}
    print(split,": Images =",len(images),"Labels =",len(labels),"Missing labels =",len(image_names-label_names))

train : Images = 1892 Labels = 1879 Missing labels = 13
valid : Images = 179 Labels = 179 Missing labels = 0
test : Images = 89 Labels = 89 Missing labels = 0


In [ ]:
import os
BASE="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8"
YAML_PATH=os.path.join(BASE,"enhanced.yaml")
yaml_content="""path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant
"""
with open(YAML_PATH,"w") as f:
    f.write(yaml_content)
print("YAML created:",YAML_PATH)
print()
print(open(YAML_PATH).read())

YAML created: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/enhanced.yaml

path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant



In [ ]:
from ultralytics import YOLO
AP_YAML="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/enhanced.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
    data=AP_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/Day17_YOLO_Enhanced",
    name="Aquatic_Plant"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/enhanced.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=

In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/weights/best.pt"
YAML_PATH="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/enhanced.yaml"
model=YOLO(MODEL_PATH)
test_results=model.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 31.8±10.4 MB/s, size: 63.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/test/labels... 89 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 329.6it/s 0.3s
val: New cache created: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.5s
                   all         89         54          1       0.94      0.987      0.719
Speed: 8.3ms preprocess, 9.8ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to /content/runs/detect/val


In [ ]:
from ultralytics import YOLO
import os
import time
MODEL_PATH="/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/valid/images"
model=YOLO(MODEL_PATH)
images=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
][:100]
start=time.time()
for image in images:
    model.predict(image,imgsz=640,device=0,verbose=False)
total_time=time.time()-start
avg_time=total_time/len(images)
fps=1/avg_time
print("Images measured:",len(images))
print("Total time:",round(total_time,2),"seconds")
print("Average time:",round(avg_time*1000,2),"ms/image")
print("Approximate FPS:",round(fps,2))

Images measured: 100
Total time: 1.39 seconds
Average time: 13.93 ms/image
Approximate FPS: 71.78


In [ ]:
summary="""# Aquatic Plant — Day 17 Enhanced YOLOv8n

## Experiment
- Dataset: Aquatic Plant
- Model: YOLOv8n
- Enhancement: White Balance + CLAHE
- Epochs: 30
- Image Size: 640 × 640
- Batch Size: 16
- Device: Tesla T4 GPU

## Validation Results

| Metric | Result |
|---|---:|
| Precision | 94.6% |
| Recall | 97.9% |
| mAP@0.5 | 98.7% |
| mAP@0.5:0.95 | 70.7% |

## Test Results

| Metric | Result |
|---|---:|
| Precision | 100.0% |
| Recall | 94.0% |
| mAP@0.5 | 98.7% |
| mAP@0.5:0.95 | 71.9% |

## Inference Performance

- Images measured: 100
- Total prediction time: 1.39 seconds
- Average prediction time: 13.93 ms/image
- Approximate FPS: 71.78

## Original vs Enhanced — Test

| Metric | Original | Enhanced |
|---|---:|---:|
| Precision | 95.69% | 100.00% |
| Recall | 94.44% | 94.00% |
| mAP@0.5 | 98.55% | 98.70% |
| mAP@0.5:0.95 | 70.63% | 71.90% |

## Saved Model

/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/weights/best.pt

## Saved Predictions

/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/predictions/validation_predictions

## Note

The enhanced model was evaluated on enhanced validation and test images. The comparison with the original model uses the same dataset splits.
"""

path="/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/Aquatic_Plant_Day17_Summary.md"

with open(path,"w") as f:
    f.write(summary)

print("Aquatic Plant Day 17 summary saved successfully!")
print(path)

Aquatic Plant Day 17 summary saved successfully!
/content/drive/MyDrive/Day17_YOLO_Enhanced/Aquatic_Plant/Aquatic_Plant_Day17_Summary.md


In [ ]:
import os
BASE="/content/Dataset_V1_Enhanced/well.v8i.yolov8"
for split in ["train","valid","test"]:
    images=os.path.join(BASE,split,"images")
    labels=os.path.join(BASE,split,"labels")
    if os.path.exists(images):
        image_count=len([f for f in os.listdir(images) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
        label_count=len([f for f in os.listdir(labels) if f.lower().endswith(".txt")]) if os.path.exists(labels) else 0
        print(split,": Images =",image_count,"Labels =",label_count,"Missing labels =",image_count-label_count)
    else:
        print(split,": NOT FOUND")

train : Images = 3795 Labels = 3780 Missing labels = 15
valid : NOT FOUND
test : NOT FOUND


In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
ORIGINAL="/content/drive/MyDrive/Dataset_V1"
ENHANCED="/content/Dataset_V1_Enhanced"
def white_balance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    avg_a=np.mean(a)
    avg_b=np.mean(b)
    a=np.clip(a-(avg_a-128)*l/255,0,255).astype(np.uint8)
    b=np.clip(b-(avg_b-128)*l/255,0,255).astype(np.uint8)
    lab=cv2.merge((l,a,b))
    return cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)
def apply_clahe(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=clahe.apply(l)
    enhanced=cv2.merge((l,a,b))
    return cv2.cvtColor(enhanced,cv2.COLOR_LAB2BGR)
def enhance_image(img):
    img=white_balance(img)
    img=apply_clahe(img)
    return img
dataset="well.v8i.yolov8"
for split in ["valid","test"]:
    src_images=os.path.join(ORIGINAL,dataset,split,"images")
    src_labels=os.path.join(ORIGINAL,dataset,split,"labels")
    if not os.path.exists(src_images):
        print(split,"not available in original dataset")
        continue
    dst_images=os.path.join(ENHANCED,dataset,split,"images")
    dst_labels=os.path.join(ENHANCED,dataset,split,"labels")
    os.makedirs(dst_images,exist_ok=True)
    os.makedirs(dst_labels,exist_ok=True)
    image_files=[
        f for f in os.listdir(src_images)
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
    ]
    for filename in tqdm(image_files,desc=split):
        src=os.path.join(src_images,filename)
        dst=os.path.join(dst_images,filename)
        img=cv2.imread(src)
        if img is None:
            print("Could not read:",filename)
            continue
        enhanced=enhance_image(img)
        cv2.imwrite(dst,enhanced)
    label_files=[
        f for f in os.listdir(src_labels)
        if f.lower().endswith(".txt")
    ]
    for filename in label_files:
        src=os.path.join(src_labels,filename)
        dst=os.path.join(dst_labels,filename)
        shutil.copy2(src,dst)

    print(split,"completed:",len(image_files),"images")

valid: 100%|██████████| 191/191 [00:07<00:00, 24.36it/s]


valid completed: 191 images


test: 100%|██████████| 184/184 [00:07<00:00, 25.64it/s]


test completed: 184 images


In [ ]:
import os
BASE="/content/Dataset_V1_Enhanced/well.v8i.yolov8"
for split in ["train","valid","test"]:
    images=os.path.join(BASE,split,"images")
    labels=os.path.join(BASE,split,"labels")
    image_count=len([f for f in os.listdir(images) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
    label_count=len([f for f in os.listdir(labels) if f.lower().endswith(".txt")])
    print(split,": Images =",image_count,"Labels =",label_count,"Missing labels =",image_count-label_count)

train : Images = 3795 Labels = 3780 Missing labels = 15
valid : Images = 191 Labels = 191 Missing labels = 0
test : Images = 184 Labels = 184 Missing labels = 0


In [ ]:
import os
import shutil
SOURCE="/content/drive/MyDrive/Dataset_V1_Enhanced/well.v8i.yolov8"
DEST="/content/Dataset_V1_Enhanced/well.v8i.yolov8"
if os.path.exists(DEST):
    print("Local Well enhanced dataset already exists.")
else:
    print("Copying enhanced Well dataset to Colab local storage...")
    shutil.copytree(SOURCE,DEST)
    print("Copy completed successfully!")

print("Local path:",DEST)

Local Well enhanced dataset already exists.
Local path: /content/Dataset_V1_Enhanced/well.v8i.yolov8


In [ ]:
YAML_PATH="/content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml"
yaml_content="""path: /content/Dataset_V1_Enhanced/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone
"""
with open(YAML_PATH,"w") as f:
    f.write(yaml_content)

print("YAML created successfully!")
print(YAML_PATH)

YAML created successfully!
/content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml


In [ ]:
from ultralytics import YOLO
WELL_YAML="/content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml"
model=YOLO("yolov8n.pt")
results=model.train(
    data=WELL_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/Day17_YOLO_Enhanced",
    name="Well"
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Well, nbs=64, nms

In [ ]:
from ultralytics import YOLO
MODEL_PATH="/content/drive/MyDrive/Day17_YOLO_Enhanced/Well/weights/best.pt"
YAML_PATH="/content/Dataset_V1_Enhanced/well.v8i.yolov8/enhanced.yaml"
model=YOLO(MODEL_PATH)
test_results=model.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 19.0±14.4 MB/s, size: 67.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/Dataset_V1_Enhanced/well.v8i.yolov8/test/labels... 184 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 184/184 331.8it/s 0.6s
val: New cache created: /content/Dataset_V1_Enhanced/well.v8i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.8it/s 4.2s
                   all        184       2111      0.431      0.394      0.411      0.181
                fishes        161       2065      0.707       0.67      0.692       0.26
        school-of-fish         23         43      0.58

In [ ]:
from ultralytics import YOLO
import os
import time
MODEL_PATH="/content/drive/MyDrive/Day17_YOLO_Enhanced/Well/weights/best.pt"
IMAGE_DIR="/content/Dataset_V1_Enhanced/well.v8i.yolov8/valid/images"
model=YOLO(MODEL_PATH)
image_files=[
    os.path.join(IMAGE_DIR,f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
][:100]
start=time.time()
for image_path in image_files:
    model.predict(
        source=image_path,
        imgsz=640,
        device=0,
        verbose=False
    )
total_time=time.time()-start
avg_time=total_time/len(image_files)
fps=1/avg_time
print("Images measured:",len(image_files))
print("Total prediction time:",round(total_time,2),"seconds")
print("Average prediction time:",round(avg_time*1000,2),"ms/image")
print("Approximate FPS:",round(fps,2))

Images measured: 100
Total prediction time: 1.58 seconds
Average prediction time: 15.82 ms/image
Approximate FPS: 63.23


In [ ]:
from IPython.display import display
import pandas as pd
import os

save_dir="/content/drive/MyDrive/Day17_YOLO_Enhanced/Well"
os.makedirs(save_dir,exist_ok=True)

validation_data={
    "Metric":["Precision","Recall","mAP@0.5","mAP@0.5:0.95"],
    "Original":[35.6,36.8,36.5,16.4],
    "Enhanced":[36.3,37.7,35.2,15.7]
}

validation_df=pd.DataFrame(validation_data)
validation_df["Change (pp)"]=validation_df["Enhanced"]-validation_df["Original"]

print("WELL — VALIDATION RESULTS")
display(validation_df)

test_df=pd.DataFrame({
    "Metric":["Precision","Recall","mAP@0.5","mAP@0.5:0.95"],
    "Enhanced Test":[43.1,39.4,41.1,18.1]
})

print("WELL — TEST RESULTS")
display(test_df)

class_df=pd.DataFrame({
    "Class":["Inlet-pipe","fishes","school-of-fish","stone"],
    "Precision":["—",70.7,58.6,0.0],
    "Recall":["—",67.0,51.2,0.0],
    "mAP@0.5":["—",69.2,54.1,0.0],
    "mAP@0.5:0.95":["—",26.0,28.2,0.0]
})

print("WELL — TEST CLASS-WISE RESULTS")
display(class_df)

summary=f"""# Well — YOLOv8n Enhanced Detection — Day 17

## Dataset
- Training images: 3,795
- Validation images: 191
- Test images: 184
- Number of classes: 4
- Classes: Inlet-pipe,fishes,school-of-fish,stone

## Model Configuration
- Model: YOLOv8n
- Epochs: 30
- Image Size: 640 × 640
- Batch Size: 16
- Device: Tesla T4 GPU
- Enhancement: White Balance + CLAHE

## Validation Performance

| Metric | Original | Enhanced | Change |
|---|---:|---:|---:|
| Precision | 35.6% | 36.3% | +0.7 pp |
| Recall | 36.8% | 37.7% | +0.9 pp |
| mAP@0.5 | 36.5% | 35.2% | -1.3 pp |
| mAP@0.5:0.95 | 16.4% | 15.7% | -0.7 pp |

## Enhanced Test Set Performance

- Test images: 184
- Test instances: 2,111

| Metric | Enhanced Test |
|---|---:|
| Precision | 43.1% |
| Recall | 39.4% |
| mAP@0.5 | 41.1% |
| mAP@0.5:0.95 | 18.1% |

## Test Class-wise Evaluation

| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|---|---:|---:|---:|---:|
| Inlet-pipe | — | — | — | — |
| fishes | 70.7% | 67.0% | 69.2% | 26.0% |
| school-of-fish | 58.6% | 51.2% | 54.1% | 28.2% |
| stone | 0.0% | 0.0% | 0.0% | 0.0% |

## Inference Performance

- Images measured: 100
- Total prediction time: 1.58 seconds
- Average prediction time: 15.82 ms/image
- Approximate FPS: 63.23

## Model Information

- Parameters: 3,006,428
- GFLOPs: 8.1
- Best model:
  `/content/drive/MyDrive/Day17_YOLO_Enhanced/Well/weights/best.pt`

## Prediction Outputs

- Validation prediction outputs:
  `/content/drive/MyDrive/Day17_YOLO_Enhanced/Well/predictions/validation_predictions`

## Findings

- The Well dataset shows substantially lower detection performance than the other two datasets.
- The original and enhanced validation results are similar.
- White Balance + CLAHE slightly increased Precision and Recall,but mAP@0.5 and mAP@0.5:0.95 decreased slightly.
- The test set contains strong class imbalance,with 2,065 of 2,111 instances belonging to the fishes class.
- The stone class has only 3 test instances and was not detected.
- Inlet-pipe does not have a reported test metric in the evaluation output.

## Tracking

No video or sequential data is currently available. Therefore,tracking was not implemented.

## Conclusion

For the Well dataset,White Balance + CLAHE did not produce an overall improvement in YOLOv8n detection performance compared with the original-image baseline. The experiment demonstrates that the effect of underwater image enhancement is dataset-dependent.
"""

summary_path=os.path.join(save_dir,"Well_Day17_Summary.md")

with open(summary_path,"w") as f:
    f.write(summary)

print("Summary saved to:")
print(summary_path)

WELL — VALIDATION RESULTS


,Metric,Original,Enhanced,Change (pp)
0,Precision,35.6,36.3,0.7
1,Recall,36.8,37.7,0.9
2,mAP@0.5,36.5,35.2,-1.3
3,mAP@0.5:0.95,16.4,15.7,-0.7


WELL — TEST RESULTS


,Metric,Enhanced Test
0,Precision,43.1
1,Recall,39.4
2,mAP@0.5,41.1
3,mAP@0.5:0.95,18.1


WELL — TEST CLASS-WISE RESULTS


,Class,Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Inlet-pipe,—,—,—,—
1,fishes,70.7,67.0,69.2,26.0
2,school-of-fish,58.6,51.2,54.1,28.2
3,stone,0.0,0.0,0.0,0.0


Summary saved to:
/content/drive/MyDrive/Day17_YOLO_Enhanced/Well/Well_Day17_Summary.md


In [ ]:
import pandas as pd
comparison_data=[
    ["DIATAquarium","Original",90.02,92.09,92.72,65.90],
    ["DIATAquarium","Enhanced",87.70,90.20,91.10,64.00],
    ["Aquatic Plant","Original",94.64,93.52,98.05,70.79],
    ["Aquatic Plant","Enhanced",94.60,97.90,98.70,70.70],
    ["Well","Original",35.60,36.80,36.50,16.40],
    ["Well","Enhanced",36.30,37.70,35.20,15.70]
]
df=pd.DataFrame(
    comparison_data,
    columns=[
        "Dataset",
        "Version",
        "Precision (%)",
        "Recall (%)",
        "mAP@0.5 (%)",
        "mAP@0.5:0.95 (%)"
    ]
)
display(df)

,Dataset,Version,Precision (%),Recall (%),mAP@0.5 (%),mAP@0.5:0.95 (%)
0,DIATAquarium,Original,90.02,92.09,92.72,65.90
1,DIATAquarium,Enhanced,87.70,90.20,91.10,64.00
2,Aquatic Plant,Original,94.64,93.52,98.05,70.79
3,Aquatic Plant,Enhanced,94.60,97.90,98.70,70.70
4,Well,Original,35.60,36.80,36.50,16.40
5,Well,Enhanced,36.30,37.70,35.20,15.70


In [ ]:
original=df[df["Version"]=="Original"].set_index("Dataset")
enhanced=df[df["Version"]=="Enhanced"].set_index("Dataset")
change=pd.DataFrame({
    "Precision Change (pp)":enhanced["Precision (%)"]-original["Precision (%)"],
    "Recall Change (pp)":enhanced["Recall (%)"]-original["Recall (%)"],
    "mAP@0.5 Change (pp)":enhanced["mAP@0.5 (%)"]-original["mAP@0.5 (%)"],
    "mAP@0.5:0.95 Change (pp)":enhanced["mAP@0.5:0.95 (%)"]-original["mAP@0.5:0.95 (%)"]
})
display(change.round(2))

,Precision Change (pp),Recall Change (pp),mAP@0.5 Change (pp),mAP@0.5:0.95 Change (pp)
Dataset,,,,
DIATAquarium,-2.32,-1.89,-1.62,-1.90
Aquatic Plant,-0.04,4.38,0.65,-0.09
Well,0.70,0.90,-1.30,-0.70


In [ ]:
save_path="/content/drive/MyDrive/Day17_YOLO_Enhanced/Day17_Original_vs_Enhanced_Comparison.csv"
df.to_csv(save_path,index=False)
print("Comparison table saved:")
print(save_path)

Comparison table saved:
/content/drive/MyDrive/Day17_YOLO_Enhanced/Day17_Original_vs_Enhanced_Comparison.csv


In [ ]:
import pandas as pd
speed_data=[
    ["DIATAquarium","Original",21.25,47.05],
    ["DIATAquarium","Enhanced",29.96,33.38],
    ["Aquatic Plant","Original",12.60,79.35],
    ["Aquatic Plant","Enhanced",13.93,71.78],
    ["Well","Original",18.55,53.91],
    ["Well","Enhanced",15.82,63.23]
]
speed_df=pd.DataFrame(
    speed_data,
    columns=["Dataset","Version","Average Time (ms/image)","FPS"]
)
display(speed_df)

,Dataset,Version,Average Time (ms/image),FPS
0,DIATAquarium,Original,21.25,47.05
1,DIATAquarium,Enhanced,29.96,33.38
2,Aquatic Plant,Original,12.60,79.35
3,Aquatic Plant,Enhanced,13.93,71.78
4,Well,Original,18.55,53.91
5,Well,Enhanced,15.82,63.23


In [ ]:
original=speed_df[speed_df["Version"]=="Original"].set_index("Dataset")
enhanced=speed_df[speed_df["Version"]=="Enhanced"].set_index("Dataset")
speed_change=pd.DataFrame({
    "Inference Time Change (ms)":enhanced["Average Time (ms/image)"]-original["Average Time (ms/image)"],
    "FPS Change":enhanced["FPS"]-original["FPS"]
})
display(speed_change.round(2))

,Inference Time Change (ms),FPS Change
Dataset,,
DIATAquarium,8.71,-13.67
Aquatic Plant,1.33,-7.57
Well,-2.73,9.32


In [ ]:
save_path="/content/drive/MyDrive/Day17_YOLO_Enhanced/Day17_Inference_Speed_Comparison.csv"
speed_df.to_csv(save_path,index=False)
print("Saved:")
print(save_path)

Saved:
/content/drive/MyDrive/Day17_YOLO_Enhanced/Day17_Inference_Speed_Comparison.csv


# Day 17 — Final Findings and Conclusion

## Overall Findings

- White Balance + CLAHE was evaluated with YOLOv8n across all three supplied underwater datasets.
- The effect of image enhancement was dataset-dependent.
- DIATAquarium showed a decrease in the reported detection metrics after enhancement.
- Aquatic Plant showed an improvement in Recall and mAP@0.5,while the other metrics changed only slightly.
- Well showed small improvements in Precision and Recall,but mAP@0.5 and mAP@0.5:0.95 decreased slightly.
- Inference speed also changed differently across datasets.
- No video or sequential data was available,so object tracking was not implemented.

## Configuration Limitation

- The original and enhanced experiments used the same YOLOv8n architecture,640 × 640 image size and batch size of 16.
- DIATAquarium original training used 30 epochs,while the enhanced experiment used 20 epochs. This difference is documented as a comparability limitation.

## Conclusion

The Day 17 experiment shows that White Balance + CLAHE does not consistently improve YOLOv8n object detection across all underwater datasets. Its effect depends on the characteristics of the dataset. The original and enhanced detection results,inference performance,and class-wise behaviour were recorded for further analysis.

## Tracking Status

No video or sequential data was available. Therefore,tracking was not implemented in the current project scope.